In [6]:
import pandas as pd

# Direct link to the classic Telco Churn dataset
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

# Load the data into a table (DataFrame)
customer_data = pd.read_csv(url)

# Show the first 5 rows
customer_data.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [7]:
# Check the data types and look for missing values
customer_data.info()


<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [8]:
# 1. Drop the customerID column 
customer_data = customer_data.drop('customerID', axis=1)

# 2. Force TotalCharges to be numeric (errors='coerce' turns blanks into missing values)
customer_data['TotalCharges'] = pd.to_numeric(customer_data['TotalCharges'], errors='coerce')

# 3. Fill those missing values with 0
customer_data['TotalCharges'] = customer_data['TotalCharges'].fillna(0)

# 4. Show a success message
print("Data cleaned! TotalCharges is now a number.")

Data cleaned! TotalCharges is now a number.


In [9]:
# Translate all remaining text columns into 1s and 0s
customer_data = pd.get_dummies(customer_data, drop_first=True)

# Let's look at the first 5 rows of our fully numbered dataset!
customer_data.head()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,MultipleLines_Yes,...,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,Churn_Yes
0,0,1,29.85,29.85,False,True,False,False,True,False,...,False,False,False,False,False,True,False,True,False,False
1,0,34,56.95,1889.50,True,False,False,True,False,False,...,False,False,False,True,False,False,False,False,True,False
2,0,2,53.85,108.15,True,False,False,True,False,False,...,False,False,False,False,False,True,False,False,True,True
3,0,45,42.30,1840.75,True,False,False,False,True,False,...,False,False,False,True,False,False,False,False,False,False
4,0,2,70.70,151.65,False,False,False,True,False,False,...,False,False,False,False,False,True,False,True,False,True


In [10]:
%pip install scikit-learn


  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached narwhals-2.23.0-py3-none-any.whl.metadata (15 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   -------- ------------------------------- 1.8/8.3 MB 8.3 MB/s eta 0:00:01
   -------------------- ------------------- 4.2/8.3 MB 9.6 MB/s eta 0:00:01
   ------------------------------ --------- 6.3/8.3 MB 10.0 MB/s eta 0:00:01
   ---------------------------------------  8.1/8.3 MB 9.5 MB/s eta 0:00:01
   ---------------------------------------- 8.3/8.3 MB 9.2 MB/s  0:00:01
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached narwhals-2.23.0-py3-none-any.whl (458 kB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)

   ---------- ----------------------------- 1/4 [narwhals]
   ---------- ----------------------------- 1/4 [narwhals]
   


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
# Import the tools we just installed
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# Step 1: Separate the "Features" (Inputs) from the "Target" (What we want to predict)
# We drop the Churn column to get our inputs (X)
X = customer_data.drop('Churn_Yes', axis=1) 
# We isolate the Churn column to get our answers (y)
y = customer_data['Churn_Yes']

# Step 2: Split the data (80% for training, 20% for testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# Step 3: Create the "Brain" (The Random Forest algorithm)
model = RandomForestClassifier(random_state=42)

# Step 4: Train the model! (This is where the math happens)
model.fit(X_train, y_train)

print("Training complete! The model has learned the patterns.")

Training complete! The model has learned the patterns.


In [12]:
from sklearn.metrics import accuracy_score, confusion_matrix

# 1. Hand the model the final exam (just the customer info)
predictions = model.predict(X_test)

# 2. Grade the exam (Compare its predictions to the real answers)
accuracy = accuracy_score(y_test, predictions)
print(f"Model Accuracy: {accuracy * 100:.1f}%\n")

# 3. See exactly *how* it made mistakes
print("The Confusion Matrix (Actual vs Predicted):")
print(confusion_matrix(y_test, predictions))

Model Accuracy: 78.5%

The Confusion Matrix (Actual vs Predicted):
[[943  93]
 [210 163]]


In [13]:
import pandas as pd

# 1. Ask the model for its importance scores
importance_scores = model.feature_importances_

# 2. Match those scores to the names of our columns
feature_ranking = pd.DataFrame({
    'Warning Sign (Feature)': X.columns,
    'Importance Score': importance_scores
})

# 3. Sort the list from highest to lowest and look at the top 10
feature_ranking = feature_ranking.sort_values(by='Importance Score', ascending=False)
feature_ranking.head(10)

,Warning Sign (Feature),Importance Score
3,TotalCharges,0.192677
2,MonthlyCharges,0.172247
1,tenure,0.171681
10,InternetService_Fiber optic,0.037451
28,PaymentMethod_Electronic check,0.035764
25,Contract_Two year,0.029908
13,OnlineSecurity_Yes,0.029213
4,gender_Male,0.026934
26,PaperlessBilling_Yes,0.025771
19,TechSupport_Yes,0.023863


In [14]:
# Ask the model for the exact probabilities instead of just 1s and 0s
probabilities = model.predict_proba(X_test)

# The model gives us two numbers: [Probability of Staying, Probability of Leaving]
# We just want the second number (Probability of Leaving), which is at index 1
risk_percentages = probabilities[:, 1]

# Let's attach these percentages to our test data so we can see who the high-risk people are
results_table = pd.DataFrame({
    'Customer_Index': X_test.index,
    'Actual_Churn': y_test,
    'Risk_Percentage': risk_percentages * 100 # Multiply by 100 to make it a clean percentage
})

# Show the top 5 highest-risk customers!
results_table.sort_values(by='Risk_Percentage', ascending=False).head(5)

,Customer_Index,Actual_Churn,Risk_Percentage
6933,6933,True,100.0
4800,4800,True,100.0
5782,5782,True,100.0
5640,5640,True,97.0
970,970,True,97.0


In [15]:
# 1. Make a copy of our test data so we don't mess up the original
dashboard_data = X_test.copy()

# 2. Add our new Risk Percentages and the Actual answers to this table
dashboard_data['Risk_Percentage'] = risk_percentages * 100
dashboard_data['Actual_Churn'] = y_test

# 3. Sort the whole table to put the highest risk customers at the very top
highest_risk_first = dashboard_data.sort_values(by='Risk_Percentage', ascending=False)

# 4. Let's look at the top 5, but only show the most important warning signs to keep it clean
columns_to_show = ['Risk_Percentage', 'tenure', 'MonthlyCharges'] 
highest_risk_first[columns_to_show].head(5)

,Risk_Percentage,tenure,MonthlyCharges
6933,100.0,1,69.60
4800,100.0,1,94.00
5782,100.0,1,69.65
5640,97.0,1,79.60
970,97.0,1,90.55


In [16]:
# Save our final results table to a file in your project folder
highest_risk_first.to_csv('churn_risk_data.csv', index=False)
print("Bridge created! Data saved.")

Bridge created! Data saved.
